# Investment Decision Agent 설계/구현 문서

이 문서는 Search Agent -> Investment Decision Agent 연결 이후, 최종 보고서 노드로 전달되는 state가 어떤 기준으로 생성되는지 설명합니다.

## 1) 전체 플로우

1. Search Agent가 10개 기업 탐색
2. 시장성/기술/창업가(종합) 평가 후 과락 분류
3. 과락 통과 기업(passReport)만 Investment Decision Agent로 전달
4. Investment Decision Agent가 항목별 가중치 + DNA 유사도 점수 계산
5. 총점 내림차순 랭킹 생성 후 보고서 노드로 전달

## 2) State 스키마 핵심

입력(input):
- allRejected: bool
- passReport: list[PassedCompany]
- rejectionReport: list[PassedCompany]

출력(output):
- allRejected: bool
- rankings: list[InvestmentRanking]
- rejectionReport: list[PassedCompany]

InvestmentRanking에는 아래 보고서 필드를 포함합니다.
- rank, companyName, totalScore
- teamScore/marketScore/techScore/dnaScore(항목별 근거 포함)
- references, startupInfo

## 3) 점수 계산 로직

- Team: 35점 = startupScore(0~10) / 10 * 35
- Market: 25점 = marketScore(0~10) / 10 * 25
- Tech: 25점 = techScore(0~10) / 10 * 25
- DNA: 15점 = rolemodel cosine similarity 평균 * 15

최종 총점:

$$\text{Total} = \text{Team} + \text{Market} + \text{Tech} + \text{DNA}$$

총점 기준 내림차순 정렬 후 rank를 1부터 재부여합니다.

## 4) DNA RAG 설계 의사결정

- Qdrant 인스턴스는 기존과 공유하고, 컬렉션만 분리(dna_rolemodel)
- 롤모델 기업: NVIDIA, Qualcomm, AMD
- searchCorp와 동일한 2단계 LLM 패턴(웹검색 -> JSON 정규화)으로 StartupInfo 관점 데이터 수집
- 컬렉션에 기업 정보가 있으면 재수집하지 않음(초기 1회 세팅)
- OPENAI_API_KEY가 없으면 롤모델 수집을 스킵하고 DNA는 0점 폴백

In [ ]:
# Search -> Investment 연결 테스트 (저장된 검색 결과 기반)
# 실행 위치: main/searchCorp

import json
from agents.search_agent import _send_to_next_stage

with open('aanalysis_results.json', 'r', encoding='utf-8') as f:
    saved = json.load(f)

state = {
    'input': {
        'searchCriteria': {
            'targetDomain': 'AI반도체',
            'targetStage': '',
            'targetRegion': '',
            'fetchCount': 10,
            'excludeList': [],
        }
    },
    'output': saved['output'],
}

_send_to_next_stage(state, state['output'].get('analyses', {}))

investment = state['output'].get('investmentDecision', {})
print('allRejected:', investment.get('allRejected'))
print('rankings_count:', len(investment.get('rankings', [])))
if investment.get('rankings'):
    top = investment['rankings'][0]
    print('top_company:', top.get('companyName'))
    print('top_total:', top.get('totalScore'))